In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from datasets import load_dataset
import torch
import re
from unsloth import FastLanguageModel
from transformers import pipeline
from scipy.stats import norm

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
from helper_functions_MLE_fitting import load_data, num_parameters, get_bounds, nll_truncnorm, truncnorm_logpdf, fit_participant, nll_truncnorm_woS, nll_truncnorm_woS_grid, fit_granularity_weights, logpmf_reported, GRIDS
from configurations import DOMAIN_CONFIG, OPTIM_CONFIG

In [3]:
# helper function for printing
def until_nth_occurrence(s, substring, n, num_chars_to_print=72):
    count = 0
    index = 0
    while index < len(s):
        index = s.find(substring, index)
        if index == -1:
            break
        count += 1
        if count == n:
            return s[index + len(substring) -num_chars_to_print:index + len(substring) + 10]

        index += len(substring)

    assert False, "should not happen"

In [4]:
%load_ext autoreload
%autoreload 2

In [5]:
# TODO careful with this
used_model_id       = 1
start_simulation_id = 1

In [6]:
DOMAIN   = "Mammals"
LLM_DATA = "../Data/Preprocessed Data/narrative_data.csv"
NLL_PATH = f'../Data/Model Outputs/log_likelihood_{DOMAIN}_TESTING_ALIGNED.pth'
EXP_DATA = f'../Data/Behavioral Data/data_analysis_{DOMAIN.lower()}.csv'

if DOMAIN == "Mammals":
    from models_mammals import *
elif DOMAIN == "Food":
    from models_food import *
else:
    from models_countries import *

In [7]:
# Load data with prompt text for LLMs
dataset = load_dataset("csv", data_files=LLM_DATA)["train"]
dataset = dataset.filter(lambda x: x["domain"] == DOMAIN)
print(dataset.shape)
dataset.to_pandas().head(3)

(48, 6)


,text,domain,participant,ID,n_training,ID_items
0,Your task is to estimate the days until female...,Mammals,0,0xhc0yxm5czl,60,"76, 9, 26, 77, 70, 59, 10, 38, 43, 41, 17, 68,..."
1,Your task is to estimate the days until female...,Mammals,1,2qewr80masfh,84,"10, 26, 77, 68, 59, 17, 9, 76, 38, 43, 70, 41,..."
2,Your task is to estimate the days until female...,Mammals,2,34tcwd0j5qgq,84,"41, 10, 76, 68, 38, 9, 77, 70, 59, 43, 26, 17,..."


In [8]:
# Load behavioral data 
df      = pd.read_csv(EXP_DATA)
df_test = df[df["training"] == 0]
print(df_test.shape)
exp_data = df_test.set_index("ID_item").iloc[:, 4:]
exp_data = exp_data.sort_index(axis=1)
exp_data.head(3)

(68, 53)


,0xhc0yxm5czl,2qewr80masfh,34tcwd0j5qgq,45v39tkqnh0b,4onztfuwyeet,6wstjp4wezt4,8f6bbs5a58p2,9g1zjvfjm5ie,9ohtsd1826k8,9y9ydj97huhc,...,u13d9hi208ge,u9zfucz7zuwn,uh7i5dewrmic,uxc38enxb0sl,uyfz2oyo6483,wl5925o0j4iv,wq9cvsm023n7,xhkz4vsry78k,xic5dq81cqld,yz6r3zc0isf5
ID_item,,,,,,,,,,,,,,,,,,,,,
1,600,600,3000,400,670,600,1500,500,570,470,...,700,670,1350,867,800,750,1285,700,980,760
2,800,590,980,1500,670,700,1000,1200,730,450,...,700,720,802,600,1200,560,450,700,2399,1200
3,1500,1500,2800,1850,1475,1400,1300,1200,700,780,...,570,1475,3520,1239,1500,1566,2400,1300,1560,750


In [9]:
temp_narr = dataset.to_pandas().set_index("ID")

sorted_cols = {}

for pid in exp_data.columns:
    n_train = int(temp_narr.loc[pid, "n_training"])
    order   = [int(x) for x in temp_narr.loc[pid, "ID_items"].split(",")]
    test_order = order[n_train:]                       # item IDs in presentation order
    sorted_cols[pid] = exp_data.loc[test_order, pid].to_numpy()
   

exp_sorted = pd.DataFrame(sorted_cols)                 # row i = i-th test trial *as presented*
exp_sorted.head(3)

,0xhc0yxm5czl,2qewr80masfh,34tcwd0j5qgq,45v39tkqnh0b,4onztfuwyeet,6wstjp4wezt4,8f6bbs5a58p2,9g1zjvfjm5ie,9ohtsd1826k8,9y9ydj97huhc,...,u13d9hi208ge,u9zfucz7zuwn,uh7i5dewrmic,uxc38enxb0sl,uyfz2oyo6483,wl5925o0j4iv,wq9cvsm023n7,xhkz4vsry78k,xic5dq81cqld,yz6r3zc0isf5
0,220,250,1400,500,296,400,550,1200,152,460,...,1300,550,250,1480,300,1300,550,300,999,1475
1,180,200,300,205,236,1400,670,1200,639,400,...,570,2190,802,160,600,357,650,600,999,1200
2,4000,600,560,400,1475,600,2800,1200,365,580,...,570,2160,2514,275,800,1500,410,800,2100,670


In [10]:
# also store item order
item_order = pd.DataFrame({pid: [int(x) for x in temp_narr.loc[pid, "ID_items"].split(",")][int(temp_narr.loc[pid, "n_training"]):]
                           for pid in exp_data.columns})

item_order.head(3)

,0xhc0yxm5czl,2qewr80masfh,34tcwd0j5qgq,45v39tkqnh0b,4onztfuwyeet,6wstjp4wezt4,8f6bbs5a58p2,9g1zjvfjm5ie,9ohtsd1826k8,9y9ydj97huhc,...,u13d9hi208ge,u9zfucz7zuwn,uh7i5dewrmic,uxc38enxb0sl,uyfz2oyo6483,wl5925o0j4iv,wq9cvsm023n7,xhkz4vsry78k,xic5dq81cqld,yz6r3zc0isf5
0,42,72,63,14,69,54,6,36,73,32,...,25,19,32,16,65,60,67,39,35,11
1,72,28,49,47,27,3,60,62,5,79,...,5,57,2,42,13,30,6,14,4,60
2,80,1,66,1,3,19,16,79,34,42,...,3,16,11,21,72,18,65,5,8,48


In [11]:
# For prompt later:
sub_df = df[df["training"] == 1].iloc[:, [1, 4]]

# Build the string
result_string = ", ".join([f"{item} (true value: {crit})" for item, crit in zip(sub_df.iloc[:, 0], sub_df.iloc[:, 1])])

print(result_string)

Red panda (true value: 550), African buffalo (true value: 1475), Wart hog (true value: 578), Tiger (true value: 1268), Bottlenosed dolphin (true value: 2831), Western gray kangaroo (true value: 670), Sugar glider (true value: 236), Stump-tailed macaque (true value: 1186), Slender loris (true value: 380), African bush elephant (true value: 4018), Nutria (true value: 152), Eurasian red squirrel (true value: 296)


In [12]:
# Define cues etc.
cues, ex_cues, ex_crit, ex_ids, test_ids, data = load_data(DOMAIN)

# `cues` rows and the Centaur NLLs are in item-ID order (test_ids); `exp_sorted` and the
# narrative text are in presentation order. This maps an item ID to its row in `cues`.
test_pos = {int(i): k for k, i in enumerate(test_ids)}

def presentation_index(pid):
    """Row indices into `cues` / `nll_centaur[p]` that put them in `pid`'s presentation order."""
    return np.array([test_pos[i] for i in item_order[pid]])

[Mammals] test items: 68, exemplars: 12, participants: 48


In [13]:
# Load negative log-likelihood of Centaur
nll_centaur = torch.stack(torch.load(NLL_PATH, weights_only=True)).float().numpy()
nll_centaur.shape

(48, 68)

In [14]:
# ALIGNMENT CHECK
# Every object is checked against a source it was NOT derived from, so a bug in the
# notebook's own reordering shows up.
#
#   exp_sorted[pid][t]        response of the t-th test trial *as presented*   (behavioural CSV)
#   narrative text            "[TEST] Item: <name> ... <<estimate>>" in presentation order
#   cues[presentation_index]  design cue rows brought into presentation order (design CSV)
#   nll_centaur[p]            Centaur NLL sorted by item ID; the raw per-trial file is in
#                             presentation order and is used as the independent reference

design  = pd.read_csv(f"../Data/Behavioral Data/design_data_{DOMAIN.lower()}.csv", sep=";", decimal=",").set_index("ID")
raw_all = torch.load(f"../Data/Model Outputs/log_likelihood_{DOMAIN}_ALL_marcelbinz-Llama-3.1-Centaur-70B-adapter.pth", weights_only=True)
test_re = re.compile(r"\[TEST\] Item:\s*(?P<item>[^.]+?)\.\s.*?<<(?P<est>\d+)>>")
N_TEST  = exp_sorted.shape[0]
problems = []

for p, pid in enumerate(exp_sorted.columns):
    row = temp_narr.loc[pid]

    # A. participant order: column p of exp_sorted must be narrative participant p,
    #    which is also row p of nll_centaur and of the raw Centaur file
    if int(row["participant"]) != p:
        problems.append(f"{pid}: exp_sorted column {p} but narrative participant {row['participant']}")
    if len(raw_all[p]) != int(row["n_training"]) + N_TEST:
        problems.append(f"{pid}: raw Centaur file row {p} has {len(raw_all[p])} trials, "
                        f"narrative says {int(row['n_training']) + N_TEST}")

    # B. trial order: narrative <<estimates>> and item names vs exp_sorted / item_order
    trials = test_re.findall(row["text"])
    if len(trials) != N_TEST:
        problems.append(f"{pid}: parsed {len(trials)} [TEST] trials from the narrative, expected {N_TEST}")
        continue
    narr_items = [t[0].strip() for t in trials]
    narr_est   = np.array([float(t[1]) for t in trials])
    x          = exp_sorted[pid].to_numpy(float)
    ok         = np.isfinite(x)                                  # over-ceiling trials are NaN in the CSV
    if not np.allclose(x[ok], narr_est[ok]):
        bad = np.flatnonzero(ok & ~np.isclose(x, narr_est))[:3]
        problems.append(f"{pid}: estimates differ from narrative at trials {bad.tolist()}")
    want_items = design.loc[item_order[pid], "item"].str.strip().tolist()
    if narr_items != want_items:
        bad = [t for t in range(N_TEST) if narr_items[t] != want_items[t]][:3]
        problems.append(f"{pid}: item names differ from design data at trials {bad} "
                        f"(narrative {narr_items[bad[0]]!r} vs design {want_items[bad[0]]!r})")

    # C. cue order: cues permuted into presentation order must equal the design rows
    idx = presentation_index(pid)
    want_cues = design.loc[item_order[pid], [f"V{i}" for i in range(1, cues.shape[1] + 1)]].to_numpy(float)
    if not np.allclose(cues[idx], want_cues):
        problems.append(f"{pid}: cues[presentation_index] do not match the design data")

    # D. Centaur order: TESTING_ALIGNED is item-ID sorted; brought back into presentation order
    #    it must line up with the last N_TEST entries of the raw (presentation-order) file.
    #    The two files come from different Centaur runs (values agree only to ~0.1 nats)
    #    this is a correlation test: right order gives r > 0.9, wrong order/participant r < 0.4.
    raw_test = raw_all[p][-N_TEST:].float().numpy()
    r_ok  = np.corrcoef(nll_centaur[p][idx], raw_test)[0, 1]
    r_bad = np.corrcoef(nll_centaur[p], raw_test)[0, 1]          # as if ALIGNED were presentation order
    if not (r_ok > 0.9 and r_ok > r_bad):
        problems.append(f"{pid}: nll_centaur[{p}] does not line up with the raw Centaur file "
                        f"(r = {r_ok:.2f} in presentation order, {r_bad:.2f} unpermuted)")

# E. global sanity
if not np.array_equal(np.sort(test_ids), test_ids):
    problems.append("test_ids (cue rows) are not in ascending item-ID order")
if list(exp_sorted.columns) != list(exp_data.columns):
    problems.append("exp_sorted and exp_data column order differ")
if nll_centaur.shape != (exp_sorted.shape[1], N_TEST):
    problems.append(f"nll_centaur shape {nll_centaur.shape}, expected {(exp_sorted.shape[1], N_TEST)}")

if problems:
    print(f"ALIGNMENT PROBLEMS ({len(problems)}):")
    print("\n".join("  " + s for s in problems))
else:
    print(f"OK: {exp_sorted.shape[1]} participants x {N_TEST} trials -- participant order, trial order, "
          "cue order and Centaur NLL order all agree")

OK: 48 participants x 68 trials -- participant order, trial order, cue order and Centaur NLL order all agree


In [15]:
AIC_centaur =  2 * 0 + 2 * nll_centaur.sum(-1).sum()
print("AIC_centaur:", AIC_centaur)

AIC_centaur: 26475.48


In [16]:
# load LLM
llm, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen3-32B-bnb-4bit',
    max_seq_length = 32768,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(llm)

==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 2. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

unsloth/Qwen3-32B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 5120, padding_idx=151669)
    (layers): ModuleList(
      (0-63): 64 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear4bit(in_features=5120, out_features=8192, bias=False)
          (k_proj): Linear4bit(in_features=5120, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=5120, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=8192, out_features=5120, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=5120, out_features=25600, bias=False)
          (up_proj): Linear4bit(in_features=5120, out_features=25600, bias=False)
          (down_proj): Linear4bit(in_features=25600, out_features=5120, bias=False)
          (act_fn): SiLUActivation()
 

In [17]:
generator = pipeline('text-generation', model=llm, tokenizer=tokenizer, device_map='auto', return_full_text=False)

In [18]:
tidy = pd.read_csv("../Data/Behavioral Data/data_tidy_combined.csv", sep=";", decimal=",")
tidy = tidy[(tidy["domain"] == "Mammals") & (tidy["phase"] == "training")]
train_est = {pid: g["est"].astype(float).dropna().to_numpy()
             for pid, g in tidy.groupby("ID")}

In [19]:
# SETUP
n_subs         = exp_sorted.shape[1]
n_trials       = exp_sorted.shape[0]
use_model_id   = 1
models         = [model_string_GCM, model_string_CAM]
threshold      = 2 #4
ub             = 10000 # Mammals ceiling

# Standardize Predictions
# mean_crit      = np.mean(ex_crit)
# std_crit       = np.std(ex_crit)
# ex_crit_std    = (ex_crit - mean_crit) / std_crit

In [20]:
# Testing & Debugging
participant  = 4
model_string = models[use_model_id]

exec(model_string)

# Select data for just this participant
x        = exp_sorted.iloc[:, participant].values # presentation order
pid      = exp_sorted.columns[participant]
p_idx    = presentation_index(pid)                # align cues + Centaur to the same order
cues_p   = cues[p_idx]
nll_c_p  = nll_centaur[participant][p_idx]
x_train  = train_est[pid]         # training-phase estimates
pi       = fit_granularity_weights(x_train)

print(f"grid: {pi}\n")

# Model fitting initialization
sigma0      = np.std(ex_crit)
x0          = np.append(abs(0.01 * np.random.randn(NUM_PARAMETERS)), sigma0)
FULL_BOUNDS = BOUNDS + [(1.0, 10 * sigma0)]

# Compute NLLs on behavioral data for this participant
res = minimize(nll_truncnorm_woS_grid, x0, args = (model, x, cues_p, ex_cues, ex_crit, pi, ub, 'sum'),
               bounds = FULL_BOUNDS, method = 'L-BFGS-B', options={'gtol': 1e-6})

# Compute single-participant trial-by-trial NLL array
p_nll = nll_truncnorm_woS_grid(res.x, model, x, cues_p, ex_cues, ex_crit, pi, ub, 'none')

if (not np.all(np.isfinite(p_nll))) or p_nll.sum() > 1e6:
    raise RuntimeError("model crashed or produced non-finite likelihood")

# Compute AIC for this specific participant
AIC = 2 * NUM_PARAMETERS + 2 * p_nll.sum()

nll_delta      = p_nll - nll_c_p

print(f"AIC: {AIC} \nnll_centaur[participant]: {nll_c_p}  \nnll_delta: {nll_delta}")

grid: [0.46268657 0.07462687 0.17910448 0.13432836 0.10447761 0.02985075
 0.01492537]

AIC: 950.0237520405079 
nll_centaur[participant]: [ 3.5625      1.375       1.0390625   4.53125     0.13183594  2.6875
  1.640625    1.828125    2.421875    4.34375     2.6875     10.875
  3.96875     7.65625     2.3125      2.40625     7.6875      2.234375
  4.15625     5.75        2.515625    2.21875     0.8515625   8.875
  1.2421875   7.875       2.84375     2.453125    0.84765625  5.34375
  9.0625      7.375       7.1875      2.21875     1.9609375   3.34375
  3.921875    2.78125     3.53125     3.953125    1.484375    6.34375
  0.73828125  2.71875     1.8046875   2.4375      0.6640625   0.72265625
  6.9375      2.6875      1.90625     0.87890625 10.625       5.25
  0.4140625   4.71875     0.96484375  1.9609375   0.953125    1.1015625
  0.31445312  0.546875    5.25        0.25976562  0.91015625  1.6796875
  0.8125      8.6875    ]  
nll_delta: [ 3.90402891  5.8592698   7.07240031  1.20357855  8.27

In [21]:
def fit_model(model_fn, k, bounds, x, cues_p, ex_cues, ex_crit, pi, ub, sigma0,
              warm=None, scales=(0.01, 0.1, 1.0, 10.0, 100.0, 1000.0), rng=np.random):
    """Best of several L-BFGS-B runs. The LLM's parameterisation is unknown, so starts are
    drawn at every order of magnitude; `warm` (the incumbent's fitted parameters, padded)
    lets a proposal that nests the incumbent start from it. Returns (res, x0_of_best)."""
    lo = np.array([b[0] for b in bounds], float); hi = np.array([b[1] for b in bounds], float)
    starts = []
    if warm is not None:
        m = min(k, len(warm))
        for fill in (0.0, 1.0):                       # 0 = additive-neutral, 1 = multiplicative-neutral
            w = np.full(k, fill); w[:m] = warm[:m]; starts.append(w)
    starts += [s * rng.standard_normal(k) for s in scales]
    best, best_x0 = None, None
    for x0 in starts:
        x0 = np.append(np.clip(x0, lo, hi), sigma0)
        res = minimize(nll_truncnorm_woS_grid, x0,
                       args=(model_fn, x, cues_p, ex_cues, ex_crit, pi, ub, 'sum'),
                       bounds=bounds + [(1.0, 10 * sigma0)], method='L-BFGS-B',
                       options={'ftol': 1e-10, 'gtol': 1e-8, 'maxiter': 5000})
        if np.isfinite(res.fun) and (best is None or res.fun < best.fun):
            best, best_x0 = res, x0
    return best, best_x0

In [ ]:
# Seperatly
# Loop through each participant on the top level
for participant in range(10,15): # range(5, n_subs):
    print(f"\n==================================================")
    print(f"OPTIMIZING MODEL FOR PARTICIPANT {participant}")
    print(f"==================================================")

    # Reset the model string back to the base model for this participant
    model_string      = models[use_model_id]
    best_aic          = float('inf')
    best_model_string = model_string
    best_p_nll        = None
    best_pred         = None
    rejected_note     = ""
    best_params       = None
    rejected_notes    = []

    # Select data for just this participant. `x` (exp_sorted) and the narrative text are in
    # presentation order; `cues` and `nll_centaur` are in item-ID order, so both are permuted
    # into presentation order here. Without this, trial t pairs one item's response with
    # another item's cues and a third item's Centaur NLL.
    x        = exp_sorted.iloc[:, participant].values
    pid      = exp_sorted.columns[participant]
    p_idx    = presentation_index(pid)
    cues_p   = cues[p_idx]
    nll_c_p  = nll_centaur[participant][p_idx]
    x_train  = train_est[pid]         # training-phase estimates
    pi       = fit_granularity_weights(x_train)
    ub       = 10000                  # Mammals ceiling

    # Hidden noise parameter of the reporting layer (never shown to the LLM)
    sigma0       = np.std(ex_crit)
    sigma_bounds = (1.0, 10 * sigma0)

    for iteration in range(6):
        try:
            # Model fitting
            exec(model_string)

            res, x0 = fit_model(model, NUM_PARAMETERS, BOUNDS, x, cues_p, ex_cues, ex_crit,
                                pi, ub, sigma0, warm=best_params)

            # Compute single-participant trial-by-trial NLL array and the latent predictions
            p_nll = nll_truncnorm_woS_grid(res.x, model, x, cues_p, ex_cues, ex_crit, pi, ub, 'none')
            pred  = np.asarray(model(res.x[:-1], cues_p, ex_cues, ex_crit), dtype=float)
            if (not np.all(np.isfinite(p_nll))) or p_nll.sum() > 1e6:
                raise RuntimeError("model crashed or produced non-finite likelihood")
            if np.any(p_nll < -1e-6):
                raise RuntimeError("model produced log-probabilities > 0 (predictions overflowed the response range)")

            # Compute AIC for this specific participant
            AIC = 2 * NUM_PARAMETERS + 2 * p_nll.sum()
            print(f"Participant {participant}: Iteration {iteration} - AIC: {AIC} for model:")
            print(model_string)

            # accept / reject
            unfittable = (res.nit == 0) or np.allclose(res.x, x0) or np.ptp(pred) == 0
            if iteration > 0 and (unfittable or AIC > best_aic):
                print(f"Rejected (AIC {AIC:.1f} vs best {best_aic:.1f}, nit={res.nit}); reverting.")
                rejected_notes.append(
                    "\n\nNote: the following previous proposal was rejected because it fitted worse"
                    + (" and the optimizer could not move its parameters (non-smooth output)" if unfittable else "")
                    + ". Do not propose it again:\n" + model_string
                )
                model_string = best_model_string
            else:
                best_aic, best_model_string = AIC, model_string
                best_p_nll, best_pred       = p_nll, pred     # fit of the accepted model
                best_params = res.x[:-1],



        except Exception as e:
            print(f"Execution error with LLM code: {e}. Rolling back...")
            rejected_notes.append("\n\nNote: the following previous proposal was rejected because it raised an error "
                             f"({e}). Do not propose it again:\n" + model_string)
            model_string  = best_model_string

        if best_p_nll is None:
            print("Seed model could not be fitted for this participant; skipping.")
            break

        # Scientific Regret Minimization (Comparing participant against their own Centaur baseline).
        # Always computed from the currently accepted model, so the trials listed in the prompt
        # belong to the code the prompt shows (not to a proposal that was just rejected).
        pred           = best_pred
        nll_delta      = best_p_nll - nll_c_p
        srm_string     = ""
        num_srm_points = 0

        # Load prompt for participant
        participant_data = dataset.filter(lambda example: example['participant'] == participant)
        n_train_items    = participant_data[0]['n_training']

        # --- participant-level descriptives that a trial list cannot convey ---
        n_distinct   = len(np.unique(x))
        share_mult10 = np.mean(x % 10 == 0)
        share_repeat = np.mean([x[t] in x[:t] for t in range(len(x))])

        # Gather data points where model doesn't match behavior
        for trial_id in range(len(nll_delta)):
            if nll_delta[trial_id].item() > threshold:
                num_srm_points += 1
                substring = until_nth_occurrence(participant_data[0]['text'], "<<", trial_id + n_train_items + 1, 95)
                match    = re.search(r"\[TEST\] Item:\s*(?P<Item>[^.]+)\..*?<<(?P<Estimate>\d+)>>", substring)
                item     = match.group('Item')
                estimate = match.group('Estimate')
                new_string = f"Item {item}. Model predicted: {pred[trial_id]:.0f}. Participant estimated: {estimate}."
                srm_string += '* ' + new_string + '\n'

        print("num_srm_points passed to prompt:", num_srm_points)

        prompt = (
            "I am studying human behavior in an estimation experiment.\n"
            "In this experiment, participants estimate the number of days until female maturity for various mammals.\n\n"
            "Experiment Structure:\n"
            "1. Training Phase: Participants repeatedly estimated the maturity days for 12 exemplar mammals. "
            "After each estimate, they received immediate feedback showing the true value.\n"
            "The 12 training exemplars and their true values are: Red panda (550), African buffalo (1475), Wart hog (578), "
            "Tiger (1268), Bottlenosed dolphin (2831), Western gray kangaroo (670), Sugar glider (236), Stump-tailed macaque (1186), "
            "Slender loris (380), African bush elephant (4018), Nutria (152), Eurasian red squirrel (296).\n"
            "2. Testing Phase: Participants estimated the maturity days for 80 mammals in a randomized order without any feedback.\n\n"

            "Data available to the model:\n"
            "`cues` is a matrix with one row per test trial and 10 numeric feature values per mammal. "
            "`ex_cues` and `ex_crit` contain the same 10 features and the true values for the 12 training exemplars. "
            "A model may use any, all, or none of these inputs; a good model does not have to use the exemplars or the feature space in any particular way.\n\n"

            "How the model is evaluated:\n"
            "The function returns a continuous latent estimate in days. A separate response model, which you do not need to write, "
            "already accounts for participants rounding their answers and reusing round numbers, and adds response noise. "
            "Therefore the model must NOT round, floor, clip, threshold, or snap its output, and must not use argmin/argmax to pick a single exemplar: "
            "the function is fitted by gradient-based optimization and must be a smooth, differentiable function of `parameters`. "
            "Use soft operations instead (weighted sums, softmax over similarities, exp/log, softplus).\n\n"

            "I have the following computational model that is currently my best guess for how this participant made estimates in this experiment:\n"
        )

        prompt += model_string
        prompt += "".join(rejected_notes)
        prompt += (
            '\n\nThis model captures the participant reasonably well overall, but on the following trials it fails clearly compared to a baseline model. '
            'For each trial I list the item, what the model predicted with its best-fitting parameters, and what the participant actually answered:\n\n'
        )
        prompt += srm_string

        prompt += (
            '\n\nCan you suggest an improved model that captures this participant\'s behavior on the listed trials?\n'
            'Look at the pattern of discrepancies between predictions and answers before deciding what to change. '
            'The improvement can be a change to how the estimate is formed from the inputs, a change to how an internal estimate is turned into a reported number '
            '(e.g. a nonlinear response function, compression toward a typical value, bounded or coarse responses), '
            'a simpler strategy that ignores some of the inputs, or a combination. Prefer the simplest mechanism that explains the listed trials; do not add parameters that the listed trials do not motivate.\n'
            'The improvement can be a change to how the estimate is formed from the inputs, a smooth nonlinear transformation of an internal estimate '
            '(e.g. log-space regression, compression toward a typical value), a simpler strategy that uses fewer inputs, or a combination. '
            'Please structure your answer as follows:\n'
            '* CRITICAL: Keep the function signature exactly the same (`def model(parameters, cues, ex_cues=None, ex_crit=None):`).\n'
            '* Ensure that EVERY parameter defined in NUM_PARAMETERS is explicitly sliced and used in the code.\n'
            '* State the number of free parameters before the model function using the NUM_PARAMETERS variable.\n'
            '* State the bounds of the free parameters before the model function using the BOUNDS variable; the list must have exactly NUM_PARAMETERS tuples.\n'
            '* Do not change the docstring.\n'
            '* Output ONLY valid, executable Python code. Do not wrap the code in markdown blocks (e.g., no ```python or ```).\n'
            '* Do not write any text besides the code and do not elaborate any further.'
        )

        # Save output state per participant iteration
        np.savez('../Data/Model Outputs/srm_model_' + str(use_model_id) + 'p_' + str(participant) + '_iteration_' + str(iteration) + '.npz', nll=best_p_nll, num_parameters=NUM_PARAMETERS, prompt=prompt, model_string=model_string)

        messages = [
             {"role": "system", "content": ""},
             {"role": "user", "content": prompt},
        ]

        model_outputs = generator(messages, do_sample=True, temperature=0.85, top_p=0.95, top_k=40, min_p=0.1, return_full_text=False, max_new_tokens=16384)

        model_string = model_outputs[0]['generated_text'].split("</think>")[-1].strip()
        model_string = re.sub(r"^```(?:python)?\s*|\s*```$", "", model_string).strip()


OPTIMIZING MODEL FOR PARTICIPANT 10


Passing `generation_config` together with generation-related arguments=({'top_p', 'do_sample', 'temperature', 'pad_token_id', 'top_k', 'min_p', 'cache_implementation', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Participant 10: Iteration 0 - AIC: 751.9969000216062 for model:


NUM_PARAMETERS = 11
BOUNDS         = [(-5000, 5000)] * (10 + 1) 

def model(parameters, cues, ex_cues=None, ex_crit=None):
    """
    Compute predicted criterion estimate.

    Parameters
    ----------
    parameters : np.ndarray of shape (num_parameters,)
        Model parameters. intercept first, then one weight per cue

    cues : np.ndarray of shape  (n_trials, n_dim)
        Feature matrix of stimuli accross trials

    Returns
    -------
    pred_crit : np.ndarray of shape (n_trials,)
        Predicted criterion estimates for each trial
    """
    
    n_dim      = cues.shape[1]
    w          = parameters[:n_dim + 1]      # intercept + weights        

    pred_crit  = w[0] + cues @ w[1:]

    return pred_crit
num_srm_points passed to prompt: 32


Both `max_new_tokens` (=16384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/pfs/data6/home/ma/ma_ma/ma_dizydorc/ASMR/lib64/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/pfs/data6/home/ma/ma_ma/ma_dizydorc/ASMR/lib64/python3.11/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warning

Execution error with LLM code: setting an array element with a sequence. The requested array would exceed the maximum number of dimension of 1.. Rolling back...
num_srm_points passed to prompt: 32
Execution error with LLM code: setting an array element with a sequence. The requested array would exceed the maximum number of dimension of 1.. Rolling back...
num_srm_points passed to prompt: 32


Both `max_new_tokens` (=16384) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [21]:
# All participants
n_subs         = exp_data.shape[1]
n_trials       = exp_data.shape[0]
use_model_id   = 0
models         = [model_string_GCM, model_string_CAM]
simulation     = 9
threshold      = 5
best_aic       = float('inf')
last_valid_model_string = models[use_model_id]

while simulation < 10:
    nll            = np.zeros(nll_centaur.shape)
    NUM_PARAMETERS = None
    model          = None
    BOUNDS         = None
    model_string   = models[use_model_id]

    for iteration in range(5):
        try:
            # Model fitting
            exec(model_string)
    
            for participant in range(n_subs):
                # select data
                x = exp_data.iloc[:, participant].values
            
                # model fitting
                x0 = abs(0.01 * np.random.randn(NUM_PARAMETERS))
                
                # Compute NLLs on behavioral data
                res = minimize(nll_truncnorm,
                            x0,
                            args = (model, x, cues, ex_cues, ex_crit, 10000, 'sum'),
                            #bounds = BOUNDS,
                            method = 'BFGS',
                            options={'gtol': 1e-6})
                
                nll[participant] = nll_truncnorm(res.x, model, x, cues, ex_cues, ex_crit, 10000, 'none')
            
            AIC = 2 * NUM_PARAMETERS + 2 * nll.sum(-1)
            current_aic = AIC.sum()
            print(f"Iteration {iteration} - AIC: {current_aic}")
    
            if iteration > 0 and current_aic > (best_aic * 1.5):
                    print(f"Model degraded severely! Rolling back to previous best version.")
                    model_string = last_valid_model_string
                    # Adjust generation parameters slightly to force creative variance
                    temperature_setting = 0.8  
                    continue
            else:
                    best_aic                = current_aic
                    last_valid_model_string = model_string
                    temperature_setting     = 0.6

        except Exception as e:
            print(f"Execution error with LLM code: {e}. Rolling back...")
            model_string = last_valid_model_string
            continue

        # Scientific Regret Minimization
        nll_delta      = nll - nll_centaur
        srm_string     = ""
        num_srm_points = 0

        for participant in range(n_subs):
            # load prompt for participant
            participant_data = dataset.filter(lambda example: example['participant'] == participant)
            n_train_items    = participant_data[0]['n_training']
            
            # print data points
            for trial_id in range(nll_delta.shape[1]):
                substring = until_nth_occurrence(participant_data[0]['text'], "<<", trial_id + n_train_items + 1,95)
                if nll_delta[participant, trial_id].item() > threshold:
                    num_srm_points += 1
                    match    = re.search(r"\[TEST\] Item:\s*(?P<Item>[^.]+)\..*?<<(?P<Estimate>\d+)>>", substring)
                    item     = match.group('Item')
                    estimate = match.group('Estimate')
                    new_string = f"Item {item}. Participant {participant} estimated: {estimate}."
                    srm_string += '* ' + new_string + '\n'
                    
        print("num_srm_points:", num_srm_points)
        
        prompt = (
            "I am studying human behavior in an estimation experiment.\n"
            "In this experiment, participants estimate the number of days until female maturity for various mammals.\n\n"
            "Experiment Structure:\n"
            "1. Training Phase: Participants repeatedly estimated the maturity days for 12 specific exemplar mammals. "
            "After each estimate, they received immediate feedback showing the true value.\n"
            "The 12 learned training exemplars and their true values are: Red panda (true value: 550), African buffalo (true value: 1475), Wart hog (true value: 578),"
            "Tiger (true value: 1268), Bottlenosed dolphin (true value: 2831), Western gray kangaroo (true value: 670), Sugar glider (true value: 236), Stump-tailed macaque (true value: 1186),"
            "Slender loris (true value: 380), African bush elephant (true value: 4018), Nutria (true value: 152), Eurasian red squirrel (true value: 296)\n\n"
            "2. Testing Phase: Participants estimated the maturity days for 80 mammals in a randomized order without any feedback. "
            "This included the 12 old training exemplars and 68 entirely new mammals.\n\n"
            "I have the following computational model that is currently my best guess for how people make estimates in this experiment:\n"
        )
        prompt += model_string
        prompt += '\n\nThis model does capture human behavior reasonably well overall, but there are the following data points in which it does not capture human behavior yet:\n\n'
        prompt += srm_string
        prompt += '\nCan you suggest an improved model that is able to capture human behavior in the listed situations?\n'
        prompt += 'Please structure your answer as follows:\n'
        prompt += '* Keep the structure of the function exactly the same.\n'
        prompt += '* Do not include the sigma parameter anywhere else in the model, only as a second returned object since it is needed for maximum likelihood estimation.\n'
        prompt += '* Crucial: Ensure that EVERY parameter defined in NUM_PARAMETERS is explicitly sliced and used in the code. Do not leave any parameters unused.\n'
        prompt += '* Crucial: If NUM_PARAMETERS is K, ensure your bounds list has exactly K tuples, and your parameters array is sliced accurately from 0 to K-1 (with sigma as the final index parameters[-1]).\n'
        prompt += '* Do not change the docstring.\n'
        prompt += '* State the number of free parameters before the model function using the NUM_PARAMETERS variable.\n'
       # prompt += '* State the bounds of the free parameters before the model function using the BOUNDS variable.\n'
        prompt += '* Do not write any text besides that and do not elaborate any further.'
        
        np.savez('../Data/Model Outputs/srm_model_' + str(use_model_id) + '_run_' + str(simulation) + '_iteration_' + str(iteration) + '.npz', nll=nll, num_parameters=NUM_PARAMETERS, prompt=prompt, model_string=model_string)
        
        messages = [
             {"role": "system", "content": ""},
             {"role": "user", "content": prompt},
        ]
        model_outputs = generator(messages, do_sample=True, temperature=temperature_setting, top_p=0.95, top_k=20, min_p=0.1, return_full_text=False, max_new_tokens=16384)
        
        model_string = model_outputs[0]['generated_text'].split("</think>")[-1]
        print(model_string)

Iteration 0 - AIC: 6528000000001056.0
num_srm_points: 3264


ValueError: Unsloth: input length 58206 + max_new_tokens 16384 exceeds the maximum sequence length of 40960!
You will need to do long context extension by increasing the `max_seq_length` in `FastLanguageModel.from_pretrained`.